# Synthetic Data Generation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('EDA_restaurants.csv')
df.head(2)

,address,name,online_order,book_table,rate,votes,location,rest_type,cuisines,cost,listed_in(type),listed_in(city),popularity_score,appeal_score,price_tier
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,1,1,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800,Buffet,Banashankari,0.324400,0.698676,Mid-range
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,1,0,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800,Buffet,Banashankari,0.324828,0.699369,Mid-range


In [3]:
df['rest_type'].unique()


array(['Casual Dining', 'Cafe, Casual Dining', 'Quick Bites',
       'Casual Dining, Cafe', 'Cafe', 'Quick Bites, Cafe',
       'Cafe, Quick Bites', 'Delivery', 'Mess', 'Dessert Parlor',
       'Bakery, Dessert Parlor', 'Pub', 'Bakery', 'Takeaway, Delivery',
       'Fine Dining', 'Beverage Shop', 'Sweet Shop', 'Bar',
       'Beverage Shop, Quick Bites', 'Confectionery',
       'Quick Bites, Beverage Shop', 'Dessert Parlor, Sweet Shop',
       'Bakery, Quick Bites', 'Sweet Shop, Quick Bites', 'Kiosk',
       'Food Truck', 'Quick Bites, Dessert Parlor',
       'Beverage Shop, Dessert Parlor', 'Takeaway', 'Pub, Casual Dining',
       'Casual Dining, Bar', 'Dessert Parlor, Beverage Shop',
       'Quick Bites, Bakery', 'Dessert Parlor, Quick Bites',
       'Microbrewery, Casual Dining', 'Lounge', 'Bar, Casual Dining',
       'Food Court', 'Cafe, Bakery', 'Dhaba', 'Quick Bites, Sweet Shop',
       'Microbrewery', 'Food Court, Quick Bites', 'Pub, Bar',
       'Casual Dining, Pub', 'Lounge, Ba

### 1. Define Customer Personas

In [6]:
# All rest type , creating it for food explorer persona

all_rest_types = sorted({
    t.strip()
    for value in df["rest_type"].dropna()
    for t in value.split(",")
})

#DIMENSION 1: Definig persionality types and their traits

IDENTITY_SEGMENTS = {
    "Student": {
        'share':0.25,
        "age": (18, 24),
        "income": (10000, 35000),
        'cost_pref_mean':300, 'cost_pref_std':80,
        'group_size_lambda':0.7,
        'explore_prob':0.35,
        'preferred_rest_types':{
            "Quick Bites": 0.40,
            "Cafe": 0.25,
            "Dessert Parlor": 0.15,
            "Bakery": 0.10,
            "Food Court": 0.05,
            "Beverage Shop": 0.05
        }

    },
    "Office Worker": {
        "share": 0.30,
        "age": (23, 40),
        "income": (30000, 90000),
        "cost_pref_mean": 450, "cost_pref_std": 100,
        "group_size_lambda": 0.3,
        "explore_prob": 0.20,
        "preferred_rest_types": {
            "Quick Bites": 0.25,
            "Casual Dining": 0.30,
            "Cafe": 0.15,
            "Delivery": 0.20,
            "Food Court": 0.10
        }
    },
    "Family": {
        "share": 0.20,
        "age": (20, 55),
        "income": (60000, 180000),
        "cost_pref_mean": 1400, "cost_pref_std": 200,
        "group_size_lambda": 2.5,
        "explore_prob": 0.15,
        "preferred_rest_types": {
            "Casual Dining": 0.50,
            "Fine Dining": 0.20,
            "Cafe": 0.15,
            "Dessert Parlor": 0.10,
            "Food Court": 0.05
        }
    },
    "Couple": {
        "share": 0.15,
        "age": (24, 45),
        "income": (50000, 150000),
        "cost_pref_mean": 1200, "cost_pref_std": 300,
        "group_size_lambda": 1.0,
        "explore_prob": 0.30,
        "preferred_rest_types": {
             "Cafe": 0.35,
            "Fine Dining": 0.30,
            "Lounge": 0.15,
            "Pub": 0.10,
            "Microbrewery": 0.10
        }
    },
    "Food Explorer": {
        "share": 0.10,
        "age": (22, 40),                          
        "income": (40000, 120000),
        "cost_pref_mean": 600, "cost_pref_std": 250,
        "group_size_lambda": 0.4,
        "explore_prob": 0.70,
        "preferred_rest_types":
        {t: 1/len(all_rest_types) for t in all_rest_types}
    }
}

# DIMENSION 2: whos gonna churn/ churn metric
ENGAGEMENT_TIERS = {
    "power_user":  {"share": 0.10, "inter_order_days_mean": 3,  "base_hazard": 0.01},
    "regular":     {"share": 0.35, "inter_order_days_mean": 7,  "base_hazard": 0.03},
    "occasional":  {"share": 0.35, "inter_order_days_mean": 15, "base_hazard": 0.06},
    "trial_only":  {"share": 0.20, "inter_order_days_mean": 30, "base_hazard": 0.15},}


### 2. Customer Attribute Generation

**Defining Functions for:**

* Age
* Income
* Cost preference
* Group size
* Exploration probability
* Preferred restaurant type
* Location

These functions generate the actual customer attributes.

Then generating the Customers

In [7]:
# generating functions
seed = 42
rng = np.random.default_rng(seed)

# defining dimension

def assign_type(n, dimension_dict):
    names = list(dimension_dict.keys())
    prob = [dimension_dict[name]['share'] for name in names]
    return rng.choice(names, size=n, p=prob)

# defining age

def generate_age(identity_array):
    ages = []
    for s in identity_array: 
        low, high = IDENTITY_SEGMENTS[s]['age']
        ages.append(rng.integers(low, high+1))
    return np.array(ages)    

# defining income


def generate_income(identity_array):
    incomes = []
    for i in identity_array:
        low, high = IDENTITY_SEGMENTS[i]['income']
        incomes.append(rng.integers(low, high+1))
    return np.array(incomes)

# defining cost preferrance

def generate_prefered(identity_array):
    means = [IDENTITY_SEGMENTS[s]['cost_pref_mean'] for s in identity_array]
    stds = [IDENTITY_SEGMENTS[s]['cost_pref_std'] for s in identity_array]
    raw = rng.normal(loc=means, scale=stds)
    return np.clip(raw, 100, None).round(0) # using clip becuase normal could generate negative values too ,
                                            # or some values which are unrealistic for cost
                                             # here anything smaller than 100 will become 100

# defining group generati

def generate_group(identity_array):
    lambdas = np.array([IDENTITY_SEGMENTS[s]["group_size_lambda"] for s in identity_array])
    return 1 + rng.poisson(lam=lambdas)

#defining exploring_probabilities

def generate_explore_prob(identity_array):
    return np.array([IDENTITY_SEGMENTS[s]["explore_prob"] for s in identity_array])

# defining preferred rest type

def generate_pref_rest_type(identity_array):
    chosen = []
    for s in identity_array:
        prefs = IDENTITY_SEGMENTS[s]["preferred_rest_types"]
        types = list(prefs.keys())
        probs = list(prefs.values())
        chosen.append(rng.choice(types, p=probs))
    return np.array(chosen)

# generating locations

def generate_location(n, restaurants_df):
    counts = restaurants_df["location"].value_counts()
    zones = counts.index
    prob = counts / counts.sum()
    return rng.choice(zones, size=n, p=prob)

PRICE_BINS = [0, 500, 1000, 2000, float("inf")]
PRICE_LABELS = ["Budget", "Mid-range", "Premium", "Luxury"]

def cost_to_price_tier(cost_values):
    return pd.cut(cost_values, bins=PRICE_BINS, labels=PRICE_LABELS, include_lowest=True)
    

### 3. Engagement & Churn Behavior

Functions for:

* Order frequency
* Churn hazard

These are based on the engagement tiers.

In [ ]:
def generate_order_frequency(engagement_array):
    means = np.array([ENGAGEMENT_TIERS[t]["inter_order_days_mean"] for t in engagement_array])
    return rng.gamma(shape=2, scale=means / 2)

def generate_churn_hazard(engagement_array):
    return np.array([ENGAGEMENT_TIERS[t]["base_hazard"] for t in engagement_array])    

### 4. Gnerating Customers

In [ ]:
# generating customer 

n_customers = 10000 #i only generated 10k customers, more can be generated later

identity_array = assign_type(n_customers, IDENTITY_SEGMENTS)
engagement_array = assign_type(n_customers, ENGAGEMENT_TIERS)

#

age = generate_age(identity_array)
income = generate_income(identity_array)
cost_pref = generate_prefered(identity_array)
price_tier = cost_to_price_tier(cost_pref)
group_size = generate_group(identity_array)
explore_prob = generate_explore_prob(identity_array)
preferred_rest_type = generate_pref_rest_type(identity_array)
order_frequency_days = generate_order_frequency(engagement_array).round(1)
churn_hazard = generate_churn_hazard(engagement_array)
location = generate_location(n_customers, df)

customers = pd.DataFrame({
    "customer_id": [str(1000001 + i) for i in range(n_customers)],
    "identity_segment": identity_array,
    "engagement_tier": engagement_array,
    "age": age,
    "income": income,
    "cost_preference": cost_pref,
    "preferred_price_tier": price_tier,
    "group_size": group_size,
    "explore_prob": explore_prob,
    "preferred_rest_type": preferred_rest_type,
    "order_frequency_days": order_frequency_days,
    "base_churn_hazard": churn_hazard,
    "location": location,
})

customers.to_csv("customers_dummy.csv", index=False)


### 5. Saving Data

In [8]:
df=pd.read_csv("customers_dummy.csv")
df

,customer_id,identity_segment,engagement_tier,age,income,cost_preference,preferred_price_tier,group_size,explore_prob,preferred_rest_type,order_frequency_days,base_churn_hazard,location
0,1000001,Couple,occasional,43,59373,631.0,Mid-range,1,0.30,Fine Dining,27.8,0.06,Whitefield
1,1000002,Office Worker,occasional,36,87255,365.0,Budget,2,0.20,Cafe,4.2,0.06,HSR
2,1000003,Couple,regular,28,102362,902.0,Mid-range,3,0.30,Cafe,4.8,0.03,Koramangala 5th Block
3,1000004,Family,power_user,55,178855,666.0,Mid-range,7,0.15,Casual Dining,1.3,0.01,Indiranagar
4,1000005,Student,regular,24,20379,324.0,Budget,3,0.35,Beverage Shop,6.2,0.03,Bannerghatta Road
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1009996,Office Worker,trial_only,32,66042,348.0,Budget,1,0.20,Casual Dining,39.8,0.15,Koramangala 6th Block
9996,1009997,Office Worker,occasional,40,55138,464.0,Budget,1,0.20,Delivery,5.3,0.06,JP Nagar
9997,1009998,Student,trial_only,19,34321,387.0,Budget,1,0.35,Quick Bites,107.9,0.15,Electronic City
9998,1009999,Office Worker,occasional,28,66318,403.0,Budget,1,0.20,Food Court,11.9,0.06,HSR


### 7. Validation

In [10]:
# Persona/tier shares match what you set
print(customers["identity_segment"].value_counts(normalize=True).round(3))
print(customers["engagement_tier"].value_counts(normalize=True).round(3))

# Age/income ranges look sane per persona 
print(customers.groupby("identity_segment")[["age","income"]].describe())

# Cost preference ordering matches intent (Couple > Family > ... > Student)
print(customers.groupby("identity_segment")["cost_preference"].mean().sort_values())

#  No missing/null values snuck in anywhere
print(customers.isna().sum())

# Order frequency by tier makes sense 
print(customers.groupby("engagement_tier")["order_frequency_days"].mean().sort_values())

identity_segment
Office Worker    0.298
Student          0.254
Family           0.203
Couple           0.146
Food Explorer    0.098
Name: proportion, dtype: float64
engagement_tier
occasional    0.354
regular       0.349
trial_only    0.201
power_user    0.096
Name: proportion, dtype: float64
                     age                                                      \
                   count       mean        std   min   25%   50%   75%   max   
identity_segment                                                               
Couple            1457.0  34.503089   6.396161  24.0  29.0  34.0  40.0  45.0   
Family            2033.0  37.354648  10.627213  20.0  28.0  37.0  47.0  55.0   
Food Explorer      985.0  31.255838   5.429662  22.0  27.0  31.0  36.0  40.0   
Office Worker     2980.0  31.675503   5.202099  23.0  27.0  32.0  36.0  40.0   
Student           2545.0  20.920236   1.966185  18.0  19.0  21.0  23.0  24.0   

                  income                                         

### 8. Basic EDA

In [11]:
df.describe()

,customer_id,age,income,cost_preference,group_size,explore_prob,order_frequency_days,base_churn_hazard
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,1.005000e+06,30.463500,70627.203400,606.547200,1.916800,0.291830,14.048550,0.062782
std,2.886896e+03,8.750379,42275.624228,349.539606,1.272258,0.154551,14.803633,0.046627
min,1.000001e+06,18.000000,10012.000000,100.000000,1.000000,0.150000,0.000000,0.010000
25%,1.002501e+06,23.000000,33149.750000,348.000000,1.000000,0.200000,4.400000,0.030000
50%,1.005000e+06,29.000000,66317.000000,493.000000,1.000000,0.200000,9.200000,0.060000
75%,1.007500e+06,37.000000,96686.500000,804.000000,2.000000,0.350000,18.400000,0.060000
max,1.010000e+06,55.000000,179979.000000,2498.000000,10.000000,0.700000,189.200000,0.150000


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           10000 non-null  int64  
 1   identity_segment      10000 non-null  object 
 2   engagement_tier       10000 non-null  object 
 3   age                   10000 non-null  int64  
 4   income                10000 non-null  int64  
 5   cost_preference       10000 non-null  float64
 6   preferred_price_tier  10000 non-null  object 
 7   group_size            10000 non-null  int64  
 8   explore_prob          10000 non-null  float64
 9   preferred_rest_type   10000 non-null  object 
 10  order_frequency_days  10000 non-null  float64
 11  base_churn_hazard     10000 non-null  float64
 12  location              10000 non-null  object 
dtypes: float64(4), int64(4), object(5)
memory usage: 1015.8+ KB


In [13]:
df.isnull().sum()

customer_id             0
identity_segment        0
engagement_tier         0
age                     0
income                  0
cost_preference         0
preferred_price_tier    0
group_size              0
explore_prob            0
preferred_rest_type     0
order_frequency_days    0
base_churn_hazard       0
location                0
dtype: int64

In [19]:
df.groupby('engagement_tier')[['cost_preference', 'income', 'group_size', 'order_frequency_days']].mean()

,cost_preference,income,group_size,order_frequency_days
engagement_tier,,,,
occasional,606.713963,70373.933296,1.906162,14.964245
power_user,634.230290,73805.608921,2.004149,2.958610
regular,605.644699,70899.838109,1.929226,7.081891
trial_only,594.531873,69073.713147,1.872012,29.867580


In [20]:
customers.groupby("identity_segment")[
    ["cost_preference", "income", "group_size"]
].mean()

,cost_preference,income,group_size
identity_segment,,,
Couple,1202.105697,99948.701441,2.013040
Family,793.591244,121317.770290,3.536153
Food Explorer,602.881218,79247.172589,1.405076
Office Worker,452.081544,59952.019463,1.290268
Student,298.464440,22511.695874,1.499804


In [21]:
customers.groupby("identity_segment")[
    ["cost_preference", "income", "group_size"]
].agg(["mean", "std"])

cost_preference                     income                \
                            mean         std           mean           std   
identity_segment                                                            
Couple               1202.105697  302.652336   99948.701441  28098.081237   
Family                793.591244  201.336611  121317.770290  34508.655221   
Food Explorer         602.881218  242.760507   79247.172589  22834.206988   
Office Worker         452.081544  102.064472   59952.019463  17117.830848   
Student               298.464440   80.082242   22511.695874   7256.882657   

                 group_size            
                       mean       std  
identity_segment                       
Couple             2.013040  1.015928  
Family             3.536153  1.571084  
Food Explorer      1.405076  0.619282  
Office Worker      1.290268  0.542865  
Student            1.499804  0.685729